In [2]:
# ╔══════════════════════════════════════════════════════════════╗
# ║              REQUIREMENTS — run this cell first              ║
# ╚══════════════════════════════════════════════════════════════╝

import subprocess, sys

pkgs = [
    "torch-scatter",
    "torch-sparse",
    "torch-geometric",
]

# Install PyG extras that match the Kaggle torch/CUDA version
import torch
torch_ver  = torch.__version__.split("+")[0]          # e.g. "2.1.0"
cuda_ver   = "cu" + torch.version.cuda.replace(".", "") # e.g. "cu118"
pyg_url    = f"https://data.pyg.org/whl/torch-{torch_ver}+{cuda_ver}.html"

print(f"torch={torch_ver}  cuda={cuda_ver}")
print(f"Installing PyG wheels from: {pyg_url}\n")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet",
     "torch-scatter", "torch-sparse", "-f", pyg_url],
    check=True
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "torch-geometric"],
    check=True
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet",
     "scikit-learn", "pandas", "numpy", "tqdm"],
    check=True
)

print("\n✅ All requirements installed.")
print(f"   GPUs available: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"   GPU {i}: {torch.cuda.get_device_name(i)}")

torch=2.10.0  cuda=cu128
Installing PyG wheels from: https://data.pyg.org/whl/torch-2.10.0+cu128.html

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 64.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 117.6 MB/s eta 0:00:00

✅ All requirements installed.
   GPUs available: 2
   GPU 0: Tesla T4
   GPU 1: Tesla T4


In [1]:
%ls

facebook_data/  facebook_large.zip


In [2]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  MULTI-GRAPH GNN PIPELINE (MEMORY OPTIMIZED)                                ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import os, json, copy, time, warnings, logging, threading
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.utils import to_undirected, train_test_split_edges
from torch_geometric.nn import SAGEConv
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

warnings.filterwarnings("ignore")

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

# ─── CONFIG ──────────────────────────────────────────────────────────────
CFG = dict(
    feat_dim=128,
    hidden=256,
    emb_dim=128,
    num_classes=4,
    num_layers=3,
    dropout=0.3,
    lr=1e-3,
    weight_decay=5e-4,
    epochs=200,
    patience=20,
    val_ratio=0.05,
    test_ratio=0.10,
)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEV0 = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
DEV1 = torch.device("cuda:1" if torch.cuda.device_count() > 1 else DEV0)

# ─── FACEBOOK (NO DOWNLOAD) ──────────────────────────────────────────────
def load_facebook_local(base="/kaggle/working/facebook_data"):
    found = {}
    for root, _, files in os.walk(base):
        for f in files:
            full = os.path.join(root, f)
            if "edges" in f:
                found["edges"] = full
            if "target" in f:
                found["target"] = full
            if "features" in f:
                found["features"] = full

    if "edges" not in found:
        raise RuntimeError("Facebook dataset not found locally")

    log.info(f"Using existing Facebook files: {found}")
    return found

# ─── TWITTER (CACHE + FRACTION LOAD) ─────────────────────────────────────
def load_twitter_fraction(root="/kaggle/working/twitter_cache", fraction=0.2):
    try:
        from torch_geometric.datasets import SNAPDataset

        log.info("Checking cached Twitter dataset...")
        ds = SNAPDataset(root=root, name="ego-Twitter")

        log.info("Using Twitter dataset (cached or newly downloaded)")

        srcs, dsts, offset = [], [], 0

        for d in ds:
            if d.edge_index.numel() == 0:
                continue

            srcs.append(d.edge_index[0] + offset)
            dsts.append(d.edge_index[1] + offset)
            offset += d.num_nodes

            # 🔥 STOP EARLY → only load fraction
            if offset > 50000:   # HARD LIMIT (memory safe)
                break

        ei = to_undirected(torch.stack([torch.cat(srcs), torch.cat(dsts)]))
        n = ei.max().item() + 1

        log.info(f"Twitter partial graph: {n} nodes")
        return ei, n

    except Exception as e:
        log.warning(f"Twitter load failed: {e}")
        return None

# ─── FEATURES ────────────────────────────────────────────────────────────
def structural_features(num_nodes, edge_index):
    deg = torch.zeros(num_nodes)
    deg.scatter_add_(0, edge_index[0], torch.ones(edge_index.size(1)))
    deg = (deg - deg.mean()) / (deg.std() + 1e-8)

    proj = torch.randn(1, CFG["feat_dim"] // 2)
    ang = deg.unsqueeze(1) * proj

    x = torch.cat([torch.sin(ang), torch.cos(ang)], dim=1)
    return F.normalize(x, dim=1)

def degree_labels(edge_index, num_nodes):
    deg = torch.zeros(num_nodes)
    deg.scatter_add_(0, edge_index[0], torch.ones(edge_index.size(1)))

    qs = [deg.quantile(q).item() for q in np.linspace(0,1,5)[1:-1]]
    y = torch.zeros(num_nodes, dtype=torch.long)
    for i, q in enumerate(qs):
        y[deg > q] = i+1
    return y

# ─── GRAPH BUILDERS ──────────────────────────────────────────────────────
def build_facebook_graph(found):
    df = pd.read_csv(found["edges"])
    src = torch.tensor(df.iloc[:,0].values)
    dst = torch.tensor(df.iloc[:,1].values)
    ei = to_undirected(torch.stack([src, dst]))

    N = ei.max().item() + 1
    x = structural_features(N, ei)
    y = degree_labels(ei, N)

    return Data(x=x, edge_index=ei, y=y, num_nodes=N)

def build_twitter_graph(tw_result):
    if tw_result is None:
        raise RuntimeError("Twitter dataset unavailable")

    ei, N = tw_result
    x = structural_features(N, ei)
    y = degree_labels(ei, N)

    return Data(x=x, edge_index=ei, y=y, num_nodes=N)

# ─── SUBSAMPLING (CRITICAL FIX) ──────────────────────────────────────────
def subsample_graph(data, target_nodes):
    if data.num_nodes <= target_nodes:
        return data

    idx = torch.randperm(data.num_nodes)[:target_nodes]
    idx = idx.sort()[0]

    mapping = -torch.ones(data.num_nodes, dtype=torch.long)
    mapping[idx] = torch.arange(target_nodes)

    src, dst = data.edge_index
    mask = (mapping[src] >= 0) & (mapping[dst] >= 0)

    new_src = mapping[src[mask]]
    new_dst = mapping[dst[mask]]

    return Data(
        x=data.x[idx],
        edge_index=torch.stack([new_src, new_dst]),
        y=data.y[idx],
        num_nodes=target_nodes
    )

# ─── MODEL ───────────────────────────────────────────────────────────────
class GNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = SAGEConv(CFG["feat_dim"], CFG["hidden"])
        self.conv2 = SAGEConv(CFG["hidden"], CFG["hidden"])
        self.lin   = nn.Linear(CFG["hidden"], CFG["num_classes"])

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        return self.lin(x)

# ─── MAIN ────────────────────────────────────────────────────────────────
def main():

    # FACEBOOK (local)
    fb_files = load_facebook_local()
    fb_raw = build_facebook_graph(fb_files)

    # TWITTER (fraction)
    tw_result = load_twitter_fraction()
    tw_raw = build_twitter_graph(tw_result)

    # 🔥 MATCH SIZE
    target_nodes = fb_raw.num_nodes
    log.info(f"Matching Twitter to Facebook size: {target_nodes}")

    tw_raw = subsample_graph(tw_raw, target_nodes)

    # SPLIT
    fb_data = train_test_split_edges(fb_raw)
    tw_data = train_test_split_edges(tw_raw)

    # MODELS
    model0 = GNN().to(DEV0)
    model1 = GNN().to(DEV1)

    optimizer = torch.optim.Adam(model0.parameters(), lr=CFG["lr"])

    # TRAIN LOOP
    for epoch in range(1, 51):
        model0.train()
        optimizer.zero_grad()

        out = model0(fb_data.x.to(DEV0), fb_data.train_pos_edge_index.to(DEV0))
        loss = F.cross_entropy(out, fb_data.y.to(DEV0))
        loss.backward()
        optimizer.step()

        if epoch % 10 == 0:
            log.info(f"Epoch {epoch} Loss: {loss.item():.4f}")

    log.info("Training complete ✅")

# RUN
main()

2026-04-23 15:09:58,609 [INFO] Using existing Facebook files: {'target': '/kaggle/working/facebook_data/facebook_large/musae_facebook_target.csv', 'features': '/kaggle/working/facebook_data/facebook_large/musae_facebook_features.json', 'edges': '/kaggle/working/facebook_data/facebook_large/musae_facebook_edges.csv'}
2026-04-23 15:09:58,828 [INFO] Checking cached Twitter dataset...
Processing...
100%|██████████| 973/973 [02:58<00:00,  5.44it/s]
Done!
2026-04-23 15:32:36,131 [INFO] Using Twitter dataset (cached or newly downloaded)
2026-04-23 15:32:36,313 [INFO] Twitter partial graph: 50142 nodes
2026-04-23 15:32:36,353 [INFO] Matching Twitter to Facebook size: 22470
2026-04-23 15:33:25,228 [INFO] Epoch 10 Loss: 1.1850
2026-04-23 15:33:25,462 [INFO] Epoch 20 Loss: 0.8601
2026-04-23 15:33:25,691 [INFO] Epoch 30 Loss: 0.5786
2026-04-23 15:33:25,922 [INFO] Epoch 40 Loss: 0.3679
2026-04-23 15:33:26,153 [INFO] Epoch 50 Loss: 0.2303
2026-04-23 15:33:26,154 [INFO] Training complete ✅


In [6]:
import os, json, torch, numpy as np, pandas as pd
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.utils import to_undirected, train_test_split_edges
from torch_geometric.nn import SAGEConv
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# ─── DEVICE ─────────────────────────────────────────
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ─── FACEBOOK LOAD ──────────────────────────────────
def load_facebook():
    base = "/kaggle/working/facebook_data"
    for root, _, files in os.walk(base):
        for f in files:
            if "edges" in f:
                edges_path = os.path.join(root, f)

    df = pd.read_csv(edges_path)
    src = torch.tensor(df.iloc[:,0].values)
    dst = torch.tensor(df.iloc[:,1].values)
    ei = to_undirected(torch.stack([src, dst]))
    N = ei.max().item() + 1

    x = torch.randn(N, 128)  # simple features
    y = torch.randint(0, 4, (N,))  # dummy labels

    return Data(x=x, edge_index=ei, y=y, num_nodes=N)

# ─── TWITTER LOAD (CACHED) ──────────────────────────
def load_twitter():
    from torch_geometric.datasets import SNAPDataset

    ds = SNAPDataset(root="/kaggle/working/twitter_cache", name="ego-Twitter")

    srcs, dsts, offset = [], [], 0

    for d in ds:
        if d.edge_index.numel() == 0:
            continue
        srcs.append(d.edge_index[0] + offset)
        dsts.append(d.edge_index[1] + offset)
        offset += d.num_nodes

        if offset > 50000:  # limit
            break

    ei = to_undirected(torch.stack([torch.cat(srcs), torch.cat(dsts)]))
    N = ei.max().item() + 1

    x = torch.randn(N, 128)
    y = torch.randint(0, 4, (N,))

    return Data(x=x, edge_index=ei, y=y, num_nodes=N)

# ─── SUBSAMPLE ──────────────────────────────────────
def subsample(data, target):
    if data.num_nodes <= target:
        return data

    idx = torch.randperm(data.num_nodes)[:target]
    mapping = -torch.ones(data.num_nodes, dtype=torch.long)
    mapping[idx] = torch.arange(target)

    src, dst = data.edge_index
    mask = (mapping[src] >= 0) & (mapping[dst] >= 0)

    new_ei = torch.stack([mapping[src[mask]], mapping[dst[mask]]])

    return Data(
        x=data.x[idx],
        edge_index=new_ei,
        y=data.y[idx],
        num_nodes=target
    )

# ─── MODEL ──────────────────────────────────────────
class GNN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = SAGEConv(128, 256)
        self.conv2 = SAGEConv(256, 256)
        self.lin = torch.nn.Linear(256, 4)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        return self.lin(x)

# ─── LOAD DATA ──────────────────────────────────────
fb = load_facebook()
tw = load_twitter()

# match sizes
tw = subsample(tw, fb.num_nodes)

fb = train_test_split_edges(fb)
tw = train_test_split_edges(tw)

# ─── MODEL ──────────────────────────────────────────
model = GNN().to(DEV)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

# ─── TRAIN (100 epochs) ─────────────────────────────
for epoch in range(1, 101):
    model.train()
    opt.zero_grad()

    out = model(fb.x.to(DEV), fb.train_pos_edge_index.to(DEV))
    loss = F.cross_entropy(out, fb.y.to(DEV))

    loss.backward()
    opt.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch} Loss: {loss.item():.4f}")

# ─── EVALUATION ─────────────────────────────────────
def evaluate(data, name):
    model.eval()
    with torch.no_grad():
        logits = model(data.x.to(DEV), data.train_pos_edge_index.to(DEV))
        preds = logits.argmax(dim=1).cpu()
        y_true = data.y.cpu().numpy()
        proba = torch.softmax(logits, dim=1).cpu().numpy()
        acc = accuracy_score(y_true, preds)
        f1 = f1_score(y_true, preds, average="macro", zero_division=0)
        c = proba.shape[1]
        try:
            if c == 2:
                auc = roc_auc_score(y_true, proba[:, 1])
            else:
                auc = roc_auc_score(
                    y_true, proba, multi_class="ovr", average="macro", labels=np.arange(c)
                )
        except ValueError:
            auc = float("nan")
    print(f"{name} → Accuracy: {acc:.4f}, F1: {f1:.4f}, AUC: {auc:.4f}")

evaluate(fb, "Facebook")
evaluate(tw, "Twitter")


Epoch 10 Loss: 1.3680
Epoch 20 Loss: 1.3320
Epoch 30 Loss: 1.2771
Epoch 40 Loss: 1.1749
Epoch 50 Loss: 1.0244
Epoch 60 Loss: 0.8544
Epoch 70 Loss: 0.6921
Epoch 80 Loss: 0.5375
Epoch 90 Loss: 0.4088
Epoch 100 Loss: 0.3013
Facebook → Accuracy: 0.9527, F1: 0.9527
Twitter → Accuracy: 0.2479, F1: 0.2478


In [7]:
import os, json, torch, numpy as np, pandas as pd
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.utils import to_undirected, train_test_split_edges
from torch_geometric.nn import SAGEConv
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# ─── DEVICE ─────────────────────────────────────────
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ─── STRUCTURAL FEATURES (IMPORTANT) ────────────────
def structural_features(num_nodes, edge_index, dim=128):
    deg = torch.zeros(num_nodes)
    deg.scatter_add_(0, edge_index[0], torch.ones(edge_index.size(1)))
    deg = (deg - deg.mean()) / (deg.std() + 1e-8)

    proj = torch.randn(1, dim // 2)
    ang = deg.unsqueeze(1) * proj

    x = torch.cat([torch.sin(ang), torch.cos(ang)], dim=1)
    return F.normalize(x, dim=1)

# ─── DEGREE LABELS ─────────────────────────────────
def degree_labels(edge_index, num_nodes, k=4):
    deg = torch.zeros(num_nodes)
    deg.scatter_add_(0, edge_index[0], torch.ones(edge_index.size(1)))

    qs = [deg.quantile(q).item() for q in np.linspace(0,1,k+1)[1:-1]]
    y = torch.zeros(num_nodes, dtype=torch.long)

    for i, q in enumerate(qs):
        y[deg > q] = i+1

    return y

# ─── FACEBOOK LOAD ─────────────────────────────────
def load_facebook():
    base = "/kaggle/working/facebook_data"
    for root, _, files in os.walk(base):
        for f in files:
            if "edges" in f:
                edges_path = os.path.join(root, f)
            if "target" in f:
                target_path = os.path.join(root, f)

    df = pd.read_csv(edges_path)
    src = torch.tensor(df.iloc[:,0].values)
    dst = torch.tensor(df.iloc[:,1].values)
    ei = to_undirected(torch.stack([src, dst]))
    N = ei.max().item() + 1

    x = structural_features(N, ei)

    # real labels
    tdf = pd.read_csv(target_path)
    label_map = {v:i for i,v in enumerate(tdf["page_type"].unique())}
    y = torch.zeros(N, dtype=torch.long)
    for _, row in tdf.iterrows():
        y[int(row["id"])] = label_map[row["page_type"]]

    return Data(x=x, edge_index=ei, y=y, num_nodes=N)

# ─── TWITTER LOAD ─────────────────────────────────
def load_twitter():
    from torch_geometric.datasets import SNAPDataset

    ds = SNAPDataset(root="/kaggle/working/twitter_cache", name="ego-Twitter")

    srcs, dsts, offset = [], [], 0

    for d in ds:
        if d.edge_index.numel() == 0:
            continue

        srcs.append(d.edge_index[0] + offset)
        dsts.append(d.edge_index[1] + offset)
        offset += d.num_nodes

        if offset > 50000:  # limit memory
            break

    ei = to_undirected(torch.stack([torch.cat(srcs), torch.cat(dsts)]))
    N = ei.max().item() + 1

    x = structural_features(N, ei)
    y = degree_labels(ei, N)

    return Data(x=x, edge_index=ei, y=y, num_nodes=N)

# ─── SUBSAMPLE ─────────────────────────────────────
def subsample(data, target):
    if data.num_nodes <= target:
        return data

    idx = torch.randperm(data.num_nodes)[:target]
    idx = idx.sort()[0]

    mapping = -torch.ones(data.num_nodes, dtype=torch.long)
    mapping[idx] = torch.arange(target)

    src, dst = data.edge_index
    mask = (mapping[src] >= 0) & (mapping[dst] >= 0)

    new_ei = torch.stack([mapping[src[mask]], mapping[dst[mask]]])

    return Data(
        x=data.x[idx],
        edge_index=new_ei,
        y=data.y[idx],
        num_nodes=target
    )

# ─── MODEL ─────────────────────────────────────────
class GNN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = SAGEConv(128, 256)
        self.conv2 = SAGEConv(256, 256)
        self.conv3 = SAGEConv(256, 128)
        self.lin = torch.nn.Linear(128, 4)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.3, training=self.training)
        x = F.relu(self.conv2(x, edge_index))
        x = F.dropout(x, p=0.3, training=self.training)
        x = F.relu(self.conv3(x, edge_index))
        return self.lin(x)

# ─── LOAD DATA ─────────────────────────────────────
fb = load_facebook()
tw = load_twitter()

# match size
tw = subsample(tw, fb.num_nodes)

fb = train_test_split_edges(fb)
tw = train_test_split_edges(tw)

# ─── MODEL ─────────────────────────────────────────
model = GNN().to(DEV)
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=5e-4)

# ─── TRAIN ─────────────────────────────────────────
for epoch in range(1, 101):
    model.train()
    opt.zero_grad()

    out_fb = model(fb.x.to(DEV), fb.train_pos_edge_index.to(DEV))
    loss_fb = F.cross_entropy(out_fb, fb.y.to(DEV))

    out_tw = model(tw.x.to(DEV), tw.train_pos_edge_index.to(DEV))
    loss_tw = F.cross_entropy(out_tw, tw.y.to(DEV))

    loss = (loss_fb + loss_tw) / 2

    loss.backward()
    opt.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch} Loss: {loss.item():.4f}")

# ─── EVALUATE ──────────────────────────────────────
def evaluate(data, name):
    model.eval()
    with torch.no_grad():
        logits = model(data.x.to(DEV), data.train_pos_edge_index.to(DEV))
        preds = logits.argmax(dim=1).cpu()
        y_true = data.y.cpu().numpy()
        proba = torch.softmax(logits, dim=1).cpu().numpy()
        acc = accuracy_score(y_true, preds)
        f1 = f1_score(y_true, preds, average="macro", zero_division=0)
        c = proba.shape[1]
        try:
            if c == 2:
                auc = roc_auc_score(y_true, proba[:, 1])
            else:
                auc = roc_auc_score(
                    y_true, proba, multi_class="ovr", average="macro", labels=np.arange(c)
                )
        except ValueError:
            auc = float("nan")
    print(f"{name} → Accuracy: {acc:.4f}, F1: {f1:.4f}, AUC: {auc:.4f}")

evaluate(fb, "Facebook")
evaluate(tw, "Twitter")


Epoch 10 Loss: 1.2782
Epoch 20 Loss: 1.0144
Epoch 30 Loss: 0.8796
Epoch 40 Loss: 0.7743
Epoch 50 Loss: 0.7089
Epoch 60 Loss: 0.6569
Epoch 70 Loss: 0.6199
Epoch 80 Loss: 0.5965
Epoch 90 Loss: 0.5757
Epoch 100 Loss: 0.5604
Facebook → Accuracy: 0.5789, F1: 0.4699
Twitter → Accuracy: 0.9960, F1: 0.9960


In [ ]:
import os, json, torch, numpy as np, pandas as pd
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.utils import to_undirected, train_test_split_edges
from torch_geometric.nn import SAGEConv
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ─── FEATURES ─────────────────────────────────────
def structural_features(num_nodes, edge_index, dim=128):
    deg = torch.zeros(num_nodes)
    deg.scatter_add_(0, edge_index[0], torch.ones(edge_index.size(1)))
    deg = (deg - deg.mean()) / (deg.std() + 1e-8)

    proj = torch.randn(1, dim // 2)
    ang = deg.unsqueeze(1) * proj

    x = torch.cat([torch.sin(ang), torch.cos(ang)], dim=1)
    return F.normalize(x, dim=1)

# ─── LABELS ───────────────────────────────────────
def degree_labels(edge_index, num_nodes, k=4):
    deg = torch.zeros(num_nodes)
    deg.scatter_add_(0, edge_index[0], torch.ones(edge_index.size(1)))

    qs = [deg.quantile(q).item() for q in np.linspace(0,1,k+1)[1:-1]]
    y = torch.zeros(num_nodes, dtype=torch.long)

    for i, q in enumerate(qs):
        y[deg > q] = i+1

    return y

# ─── FACEBOOK ─────────────────────────────────────
def load_facebook():
    base = "/kaggle/working/facebook_data"
    for root, _, files in os.walk(base):
        for f in files:
            if "edges" in f:
                edges_path = os.path.join(root, f)
            if "target" in f:
                target_path = os.path.join(root, f)

    df = pd.read_csv(edges_path)
    src = torch.tensor(df.iloc[:,0].values)
    dst = torch.tensor(df.iloc[:,1].values)
    ei = to_undirected(torch.stack([src, dst]))
    N = ei.max().item() + 1

    x = structural_features(N, ei)

    tdf = pd.read_csv(target_path)
    label_map = {v:i for i,v in enumerate(tdf["page_type"].unique())}
    y = torch.zeros(N, dtype=torch.long)
    for _, row in tdf.iterrows():
        y[int(row["id"])] = label_map[row["page_type"]]

    return Data(x=x, edge_index=ei, y=y, num_nodes=N)

# ─── TWITTER ──────────────────────────────────────
def load_twitter():
    from torch_geometric.datasets import SNAPDataset

    ds = SNAPDataset(root="/kaggle/working/twitter_cache", name="ego-Twitter")

    srcs, dsts, offset = [], [], 0
    for d in ds:
        if d.edge_index.numel() == 0:
            continue
        srcs.append(d.edge_index[0] + offset)
        dsts.append(d.edge_index[1] + offset)
        offset += d.num_nodes
        if offset > 50000:
            break

    ei = to_undirected(torch.stack([torch.cat(srcs), torch.cat(dsts)]))
    N = ei.max().item() + 1

    x = structural_features(N, ei)
    y = degree_labels(ei, N)

    return Data(x=x, edge_index=ei, y=y, num_nodes=N)

# ─── MERGE GRAPHS ─────────────────────────────────
def merge_graphs(fb, tw):
    offset = fb.num_nodes

    tw_edge = tw.edge_index + offset

    x = torch.cat([fb.x, tw.x], dim=0)
    y = torch.cat([fb.y, tw.y], dim=0)

    edge_index = torch.cat([fb.edge_index, tw_edge], dim=1)

    return Data(x=x, edge_index=edge_index, y=y, num_nodes=x.size(0))

# ─── MODEL ────────────────────────────────────────
class GNN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = SAGEConv(128, 256)
        self.conv2 = SAGEConv(256, 256)
        self.conv3 = SAGEConv(256, 128)
        self.lin = torch.nn.Linear(128, 4)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.3, training=self.training)
        x = F.relu(self.conv2(x, edge_index))
        x = F.dropout(x, p=0.3, training=self.training)
        x = F.relu(self.conv3(x, edge_index))
        return self.lin(x)

# ─── LOAD + MERGE ─────────────────────────────────
fb = load_facebook()
tw = load_twitter()

data = merge_graphs(fb, tw)

data = train_test_split_edges(data)

# ─── TRAIN ────────────────────────────────────────
model = GNN().to(DEV)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(1, 101):
    model.train()
    opt.zero_grad()

    out = model(data.x.to(DEV), data.train_pos_edge_index.to(DEV))
    loss = F.cross_entropy(out, data.y.to(DEV))

    loss.backward()
    opt.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch} Loss: {loss.item():.4f}")

# ─── SINGLE EVALUATION ─────────────────────────────
model.eval()
with torch.no_grad():
    logits = model(data.x.to(DEV), data.train_pos_edge_index.to(DEV))
    preds = logits.argmax(dim=1).cpu()
    y_true = data.y.cpu().numpy()
    proba = torch.softmax(logits, dim=1).cpu().numpy()
    acc = accuracy_score(y_true, preds)
    f1 = f1_score(y_true, preds, average="macro", zero_division=0)
    c = proba.shape[1]
    try:
        if c == 2:
            auc = roc_auc_score(y_true, proba[:, 1])
        else:
            auc = roc_auc_score(
                y_true, proba, multi_class="ovr", average="macro", labels=np.arange(c)
            )
    except ValueError:
        auc = float("nan")

print(f"\n🔥 Combined Graph → Accuracy: {acc:.4f}, F1: {f1:.4f}, AUC: {auc:.4f}")


In [1]:
import shutil

shutil.make_archive('/kaggle/working_output', 'zip', '/kaggle/working')

'/kaggle/working_output.zip'

In [2]:
from IPython.display import FileLink
FileLink('/kaggle/working_output.zip')

/kaggle/working_output.zip

In [3]:
import os, zipfile

zip_path = '/kaggle/working_clean.zip'

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk('/kaggle/working'):
        for f in files:
            if f.endswith('.pt') or f.endswith('.npy'):  # skip large files if needed
                continue
            full = os.path.join(root, f)
            z.write(full, os.path.relpath(full, '/kaggle/working'))

print("Created:", zip_path)

Created: /kaggle/working_clean.zip


In [4]:
from IPython.display import FileLink
FileLink('/kaggle/working_clean.zip')

/kaggle/working_clean.zip

In [5]:
!mv /kaggle/working_clean.zip /kaggle/working/

In [16]:
import os, torch, numpy as np, pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.utils import to_undirected
from torch_geometric.nn import SAGEConv
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ─── FEATURES ─────────────────────────────────────
def structural_features(num_nodes, edge_index, dim=128):
    deg = torch.zeros(num_nodes)
    deg.scatter_add_(0, edge_index[0], torch.ones(edge_index.size(1)))
    deg = (deg - deg.mean()) / (deg.std() + 1e-8)

    proj = torch.randn(1, dim // 2)
    ang = deg.unsqueeze(1) * proj
    x = torch.cat([torch.sin(ang), torch.cos(ang)], dim=1)
    return F.normalize(x, dim=1)

# ─── LABELS ───────────────────────────────────────
def degree_labels(edge_index, num_nodes, k=4):
    deg = torch.zeros(num_nodes)
    deg.scatter_add_(0, edge_index[0], torch.ones(edge_index.size(1)))
    qs = [deg.quantile(q).item() for q in np.linspace(0,1,k+1)[1:-1]]

    y = torch.zeros(num_nodes, dtype=torch.long)
    for i, q in enumerate(qs):
        y[deg > q] = i+1
    return y

# ─── FACEBOOK ─────────────────────────────────────
def load_facebook():
    base = "/kaggle/working/facebook_data"
    for root, _, files in os.walk(base):
        for f in files:
            if "edges" in f: edges_path = os.path.join(root, f)
            if "target" in f: target_path = os.path.join(root, f)

    df = pd.read_csv(edges_path)
    src = torch.tensor(df.iloc[:,0].values)
    dst = torch.tensor(df.iloc[:,1].values)
    ei = to_undirected(torch.stack([src, dst]))
    N = ei.max().item() + 1

    x = structural_features(N, ei)

    tdf = pd.read_csv(target_path)
    label_map = {v:i for i,v in enumerate(tdf["page_type"].unique())}
    y = torch.zeros(N, dtype=torch.long)
    for _, row in tdf.iterrows():
        y[int(row["id"])] = label_map[row["page_type"]]

    return Data(x=x, edge_index=ei, y=y, num_nodes=N)

# ─── TWITTER LIMITED ──────────────────────────────
def load_twitter_limited(target_nodes):
    from torch_geometric.datasets import SNAPDataset

    ds = SNAPDataset(root="/kaggle/working/twitter_cache", name="ego-Twitter")

    srcs, dsts = [], []
    current_nodes, offset = 0, 0

    for d in ds:
        if current_nodes >= target_nodes: break
        if d.edge_index.numel() == 0: continue

        n = d.num_nodes
        if current_nodes + n > target_nodes:
            n = target_nodes - current_nodes
            mask = (d.edge_index[0] < n) & (d.edge_index[1] < n)
            src = d.edge_index[0][mask]
            dst = d.edge_index[1][mask]
        else:
            src, dst = d.edge_index

        srcs.append(src + offset)
        dsts.append(dst + offset)

        offset += n
        current_nodes += n

    ei = to_undirected(torch.stack([torch.cat(srcs), torch.cat(dsts)]))
    x = structural_features(current_nodes, ei)
    y = degree_labels(ei, current_nodes)

    return Data(x=x, edge_index=ei, y=y, num_nodes=current_nodes)

# ─── MERGE ───────────────────────────────────────
def merge_graphs(fb, tw):
    offset = fb.num_nodes
    tw_edge = tw.edge_index + offset
    x = torch.cat([fb.x, tw.x], dim=0)
    y = torch.cat([fb.y, tw.y], dim=0)
    edge_index = torch.cat([fb.edge_index, tw_edge], dim=1)
    return Data(x=x, edge_index=edge_index, y=y, num_nodes=x.size(0))

# ─── MODEL (DUAL TASK) ────────────────────────────
class GNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = SAGEConv(128, 256)
        self.conv2 = SAGEConv(256, 256)
        self.conv3 = SAGEConv(256, 128)

        self.node_head = nn.Linear(128, 4)
        self.link_head = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def encode(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = F.relu(self.conv3(x, edge_index))
        return x

    def forward(self, x, edge_index):
        z = self.encode(x, edge_index)
        return z, self.node_head(z)

    def predict_link(self, z, edge_index):
        pair = torch.cat([z[edge_index[0]], z[edge_index[1]]], dim=1)
        return self.link_head(pair).squeeze()

# ─── NEGATIVE SAMPLING ────────────────────────────
def negative_sampling(num_nodes, num_samples):
    src = torch.randint(0, num_nodes, (num_samples,))
    dst = torch.randint(0, num_nodes, (num_samples,))
    mask = src != dst
    return torch.stack([src[mask][:num_samples], dst[mask][:num_samples]])

# ─── LOAD ─────────────────────────────────────────
fb = load_facebook()
tw = load_twitter_limited(fb.num_nodes)
data = merge_graphs(fb, tw)

# ─── TRAIN ────────────────────────────────────────
model = GNN().to(DEV)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)

for epoch in range(1, 101):
    model.train()
    opt.zero_grad()

    z, node_logits = model(data.x.to(DEV), data.edge_index.to(DEV))

    # node loss
    node_loss = F.cross_entropy(node_logits, data.y.to(DEV), label_smoothing=0.1)

    # link loss
    pos_edges = data.edge_index
    neg_edges = negative_sampling(data.num_nodes, pos_edges.size(1))

    pos_pred = model.predict_link(z, pos_edges.to(DEV))
    neg_pred = model.predict_link(z, neg_edges.to(DEV))

    link_labels = torch.cat([
        torch.ones(pos_pred.size(0)),
        torch.zeros(neg_pred.size(0))
    ]).to(DEV)

    link_preds = torch.cat([pos_pred, neg_pred])

    link_loss = F.binary_cross_entropy(link_preds, link_labels)

    loss = 0.6 * node_loss + 0.4 * link_loss

    loss.backward()
    opt.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch} Loss: {loss.item():.4f}")

# ─── EVALUATE ─────────────────────────────────────
model.eval()
with torch.no_grad():
    _, logits = model(data.x.to(DEV), data.edge_index.to(DEV))
    preds = logits.argmax(dim=1).cpu()

    y_true = data.y.cpu().numpy()
    proba = torch.softmax(logits, dim=1).cpu().numpy()
    acc = accuracy_score(y_true, preds)
    f1 = f1_score(y_true, preds, average="macro", zero_division=0)
    c = proba.shape[1]
    try:
        if c == 2:
            auc = roc_auc_score(y_true, proba[:, 1])
        else:
            auc = roc_auc_score(
                y_true, proba, multi_class="ovr", average="macro", labels=np.arange(c)
            )
    except ValueError:
        auc = float("nan")
    acc_pct = acc * 100.0
print(f"
🔥 Combined Accuracy: {acc_pct:.4f}%, F1: {f1:.4f}, AUC: {auc:.4f}")



/usr/local/lib/python3.12/dist-packages/torch_geometric/data/in_memory_dataset.py:131: UserWarning: Weights only load failed. Please file an issue to make `torch.load(weights_only=True)` compatible in your case. Please use `torch.serialization.add_safe_globals([torch_geometric.datasets.snap_dataset.EgoData])` to allowlist this global.
  out = fs.torch_load(path)


Epoch 10 Loss: 1.0262
Epoch 20 Loss: 0.8602
Epoch 30 Loss: 0.7970
Epoch 40 Loss: 0.7526
Epoch 50 Loss: 0.7176
Epoch 60 Loss: 0.6819
Epoch 70 Loss: 0.6398
Epoch 80 Loss: 0.6013
Epoch 90 Loss: 0.5757
Epoch 100 Loss: 0.5627

🔥 Combined Accuracy: 89.9105%, F1: 0.7785
